# autoresearch — session analysis

Reads a session's logs + git and renders the search trajectory. Point it at a session via `LOGS_DIR` (default `logs`) and the repo via `REPO_DIR` (default `.`). To re-render a **finished/archived** run with correct keep/discard, set `HEAD_REF` to its branch (e.g. `autoresearch/20260601-073738-...`) and `LOGS_DIR` to its archived logs. Keep/discard is from git ancestry; intent from commit messages.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from tools.extract_decisions import extract_decisions

LOGS_DIR = os.environ.get('LOGS_DIR', 'logs')
REPO_DIR = os.environ.get('REPO_DIR', '.')
HEAD_REF = os.environ.get('HEAD_REF', 'HEAD')   # set to a run's branch to re-render an old run
FIGDIR = os.path.join(os.path.dirname(os.path.normpath(LOGS_DIR)) or '.', 'figures')
os.makedirs(FIGDIR, exist_ok=True)

df = extract_decisions(LOGS_DIR, repo_dir=REPO_DIR, head=HEAD_REF)
print(f'Loaded {len(df)} trials from {LOGS_DIR}')
df.head()

## Summary counts

In [ ]:
print('n trials   :', len(df))
print('by status  :', df['status'].value_counts().to_dict())
print('by family  :', df['model_family'].value_counts().to_dict())
print('kept (git) :', int((df['kept_on_branch'] == True).sum()))
print('discarded  :', int((df['kept_on_branch'] == False).sum()))
print('crashes    :', int((df['status'] == 'crash').sum()))

## Progress curve

Trial index vs running-best `val_logloss`; red dashed lines mark family swaps.

In [ ]:
valid = df[df['status'] != 'crash'].copy()
valid['running_best'] = valid['val_logloss'].cummin()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(valid['trial_id'], valid['val_logloss'], 'o-', alpha=0.5, label='val_logloss')
ax.plot(valid['trial_id'], valid['running_best'], 's-', color='green', label='running best')
for tid in df.loc[df['family_changed_from_prior'] == True, 'trial_id']:
    ax.axvline(tid, color='red', ls='--', alpha=0.3)
ax.set_xlabel('trial'); ax.set_ylabel('val_logloss')
ax.set_title('Progress curve (red dashed = family swap)'); ax.legend()
fig.savefig(os.path.join(FIGDIR, 'progress_curve.png'), dpi=120, bbox_inches='tight')
plt.show()

## Family distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df['model_family'].value_counts().plot.bar(ax=ax, title='Trials per family')
ax.set_ylabel('count')
fig.savefig(os.path.join(FIGDIR, 'family_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

## Locus of change

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df['locus_of_change'].value_counts(dropna=False).plot.bar(ax=ax, title='Locus of change')
ax.set_ylabel('count')
fig.savefig(os.path.join(FIGDIR, 'locus_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

## Kept vs discarded per family

Color = git-derived outcome: green kept, orange discarded, red crash.

In [ ]:
def _color(row):
    if row['status'] == 'crash':
        return 'red'
    if row['kept_on_branch'] == True:
        return 'green'
    if row['kept_on_branch'] == False:
        return 'orange'
    return 'gray'

families = list(df['model_family'].dropna().unique())
fam_idx = {f: i for i, f in enumerate(families)}
fig, ax = plt.subplots(figsize=(9, 4))
for _, r in df.iterrows():
    ax.scatter(fam_idx.get(r['model_family'], 0), r['val_logloss'],
               c=_color(r), s=60, alpha=0.7)
ax.set_xticks(range(len(families))); ax.set_xticklabels(families, rotation=30)
ax.set_ylabel('val_logloss')
ax.set_title('Kept (green) / discarded (orange) / crash (red)')
fig.savefig(os.path.join(FIGDIR, 'kept_vs_discarded.png'), dpi=120, bbox_inches='tight')
plt.show()

## Decision records

In [ ]:
df[['trial_id', 'model_family', 'locus_of_change', 'intent',
    'val_logloss', 'kept_on_branch']].head(30)